In [1]:
# Imports (already in your code, just noting)
import pandas as pd
import numpy as np
import os
import optuna
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Attention, Concatenate
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

c:\Users\khiew\anaconda3\envs\LSTM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# List all physical devices (GPUs)
physical_devices = tf.config.list_physical_devices('GPU')

# Check and print the name of the GPU being used
if len(physical_devices) > 0:
    # Setting the visible devices to use only GPU 0 (RTX 4060)
    tf.config.set_visible_devices(physical_devices[0], 'GPU')
    print(f"Using GPU: {physical_devices[0].name}")
else:
    print("No GPUs found, using CPU instead.")

Using GPU: /physical_device:GPU:0


In [3]:
# Load the single dataset
df = pd.read_csv("C:\\Users\\khiew\\Downloads\\archive (3)\\train.csv")

# Keep only short_question and short_answer
df = df[['Question', 'Answer']]

# Remove duplicated rows
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicated rows.")

# Split into training and validation sets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)  # 80% train, 20% val

# Tokenizer (fit on both questions + answers from the WHOLE dataset)
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(df['Question'].tolist() + df['Answer'].tolist())

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

Removed 48 duplicated rows.


In [4]:
# Encode Encoder Input (questions)
X_encoder_train = tokenizer.texts_to_sequences(train_df['Question'])
X_encoder_val = tokenizer.texts_to_sequences(val_df['Question'])

# Prepare Decoder Input (add <sos>)
train_df['Answer_in'] = '<sos> ' + train_df['Answer']
val_df['Answer_in'] = '<sos> ' + val_df['Answer']
X_decoder_train = tokenizer.texts_to_sequences(train_df['Answer_in'])
X_decoder_val = tokenizer.texts_to_sequences(val_df['Answer_in'])

# Prepare Decoder Output (add <eos>)
train_df['Answer_out'] = train_df['Answer'] + ' <eos>'
val_df['Answer_out'] = val_df['Answer'] + ' <eos>'
y_decoder_train = tokenizer.texts_to_sequences(train_df['Answer_out'])
y_decoder_val = tokenizer.texts_to_sequences(val_df['Answer_out'])


In [5]:
# Define maximum sequence length
max_len = 175

# Encode and pad sequences with truncation
X_encoder_train = pad_sequences(X_encoder_train, maxlen=max_len, padding='post', truncating='post')
X_encoder_val = pad_sequences(X_encoder_val, maxlen=max_len, padding='post', truncating='post')

X_decoder_train = pad_sequences(X_decoder_train, maxlen=max_len, padding='post', truncating='post')
X_decoder_val = pad_sequences(X_decoder_val, maxlen=max_len, padding='post', truncating='post')

y_decoder_train = pad_sequences(y_decoder_train, maxlen=max_len, padding='post', truncating='post')
y_decoder_val = pad_sequences(y_decoder_val, maxlen=max_len, padding='post', truncating='post')


In [6]:
def build_seq2seq_model(trial):
    # Hyperparameters
    embedding_dim = trial.suggest_int('embedding_dim', 32, 128)
    lstm_units = trial.suggest_int('lstm_units', 32, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'rmsprop'])

    # Encoder
    encoder_inputs = Input(shape=(175,))
    enc_emb = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)(encoder_inputs)
    encoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)  # return_sequences=True for attention
    encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

    # Decoder
    decoder_inputs = Input(shape=(175,))
    dec_emb = Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True)(decoder_inputs)
    decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])
    
    # Attention Layer
    attention = Attention(use_scale=True)([decoder_outputs, encoder_outputs])  # Apply attention
    context_vector = Concatenate(axis=-1)([decoder_outputs, attention])  # Concatenate the decoder output and attention context

    # Output Layer
    decoder_dense = Dense(vocab_size, activation='softmax')
    output = decoder_dense(context_vector)

    # Define model
    model = Model([encoder_inputs, decoder_inputs], output)

    # Compile model
    if optimizer_name == 'adam':
        optimizer = Adam()
    else:
        optimizer = RMSprop()
        
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model


In [7]:
def objective(trial):
    model = build_seq2seq_model(trial)

    history = model.fit(
        [X_encoder_train, X_decoder_train], 
        y_decoder_train, 
        epochs=20,
        batch_size=32,
        validation_data=([X_encoder_val, X_decoder_val], y_decoder_val),
        verbose=0
    )
    
    val_accuracy = max(history.history['val_accuracy'])
    return val_accuracy  

In [8]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="LSTM_Batch32_Token175_Epochs20LargeRangewithAttNewDataset", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/LSTM_Chatbot.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=5)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-26 23:03:22,446] Using an existing study with name 'LSTM_Batch32_Token175_Epochs20LargeRangewithAttNewDataset' instead of creating a new one.
[I 2025-04-26 23:28:20,159] Trial 3 finished with value: 0.4517042636871338 and parameters: {'dropout_rate': 0.19704583558483116, 'embedding_dim': 124, 'lstm_units': 93, 'optimizer': 'rmsprop'}. Best is trial 3 with value: 0.4517042636871338.
[I 2025-04-26 23:56:41,813] Trial 4 finished with value: 0.4435752034187317 and parameters: {'dropout_rate': 0.46524204903933575, 'embedding_dim': 42, 'lstm_units': 126, 'optimizer': 'rmsprop'}. Best is trial 3 with value: 0.4517042636871338.
[I 2025-04-27 00:26:13,869] Trial 5 finished with value: 0.45546165108680725 and parameters: {'dropout_rate': 0.4985112526264005, 'embedding_dim': 114, 'lstm_units': 106, 'optimizer': 'rmsprop'}. Best is trial 5 with value: 0.45546165108680725.
[I 2025-04-27 00:55:42,101] Trial 6 finished with value: 0.4352709949016571 and parameters: {'dropout_rate': 0.14119


Best Trial:
FrozenTrial(number=5, state=TrialState.COMPLETE, values=[0.45546165108680725], datetime_start=datetime.datetime(2025, 4, 26, 23, 56, 41, 863636), datetime_complete=datetime.datetime(2025, 4, 27, 0, 26, 13, 841402), params={'dropout_rate': 0.4985112526264005, 'embedding_dim': 114, 'lstm_units': 106, 'optimizer': 'rmsprop'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'dropout_rate': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'embedding_dim': IntDistribution(high=128, log=False, low=32, step=1), 'lstm_units': IntDistribution(high=128, log=False, low=32, step=1), 'optimizer': CategoricalDistribution(choices=('adam', 'rmsprop'))}, trial_id=54, value=None)
Best Hyperparameters:
{'dropout_rate': 0.4985112526264005, 'embedding_dim': 114, 'lstm_units': 106, 'optimizer': 'rmsprop'}
